# 28 — Prompt Architecture Patterns and System Selection

## Scenario
Your company wants to build an AI feature. The engineering team immediately proposes building a multi-agent system with memory and complex planning.

**The Problem:** Agents are slow, expensive, and unpredictable. Building an agent when a simple regex would suffice is a massive architectural failure.

**The Solution:** We must use an **Architecture Decision Matrix**. We evaluate the requirements of the task and always select the *simplest* architecture that can fulfill those requirements.

In [ ]:
from enum import Enum
from pydantic import BaseModel


## Step 1: Defining the Spectrum

We define the spectrum of AI architectures, from simple standard code up to fully autonomous agents.

In [ ]:
class Pattern(str, Enum):
    STANDARD_CODE = "Standard Code (Regex / Logic)"
    SIMPLE_PROMPT = "Simple Prompt (Zero/Few Shot)"
    RAG = "Retrieval-Augmented Generation (RAG)"
    TOOL_USE = "Tool Use / Function Calling"
    AGENT = "Autonomous Agent"

class Requirement(BaseModel):
    name: str
    is_strictly_deterministic: bool
    needs_live_data: bool
    needs_to_take_actions: bool
    is_open_ended_multi_step: bool


## Step 2: The Decision Matrix

We build a simple decision tree. Notice how it tries to exit as early as possible.

In [ ]:
def select_architecture(req: Requirement) -> Pattern:
    # 1. Can we do this without AI?
    if req.is_strictly_deterministic:
        return Pattern.STANDARD_CODE
        
    # 2. Does it need to DO anything?
    if not req.needs_to_take_actions:
        if req.needs_live_data:
            return Pattern.RAG
        else:
            return Pattern.SIMPLE_PROMPT
            
    # 3. It needs to take actions. Is it a predictable pipeline or an open-ended loop?
    if not req.is_open_ended_multi_step:
        return Pattern.TOOL_USE
    else:
        return Pattern.AGENT


## Step 3: Evaluating Scenarios

Let's pass 12 different enterprise scenarios through our decision matrix.

In [ ]:
scenarios = [
    Requirement(name="1. Extract email addresses from a text block", is_strictly_deterministic=True, needs_live_data=False, needs_to_take_actions=False, is_open_ended_multi_step=False),
    Requirement(name="2. Summarize a static PDF document", is_strictly_deterministic=False, needs_live_data=False, needs_to_take_actions=False, is_open_ended_multi_step=False),
    Requirement(name="3. Translate a French paragraph to English", is_strictly_deterministic=False, needs_live_data=False, needs_to_take_actions=False, is_open_ended_multi_step=False),
    Requirement(name="4. Answer questions using the company intranet", is_strictly_deterministic=False, needs_live_data=True, needs_to_take_actions=False, is_open_ended_multi_step=False),
    Requirement(name="5. Tell the user the current weather in Tokyo", is_strictly_deterministic=False, needs_live_data=True, needs_to_take_actions=False, is_open_ended_multi_step=False),
    Requirement(name="6. Book a flight given a specific date and time", is_strictly_deterministic=False, needs_live_data=True, needs_to_take_actions=True, is_open_ended_multi_step=False),
    Requirement(name="7. Execute a specific SQL query against the DB", is_strictly_deterministic=False, needs_live_data=True, needs_to_take_actions=True, is_open_ended_multi_step=False),
    Requirement(name="8. Research competitors, plan a marketing campaign, and email leads", is_strictly_deterministic=False, needs_live_data=True, needs_to_take_actions=True, is_open_ended_multi_step=True),
    Requirement(name="9. Autonomously fix Github issues by editing code and running tests", is_strictly_deterministic=False, needs_live_data=True, needs_to_take_actions=True, is_open_ended_multi_step=True),
    Requirement(name="10. Filter a list of profanity words", is_strictly_deterministic=True, needs_live_data=False, needs_to_take_actions=False, is_open_ended_multi_step=False),
    Requirement(name="11. Generate a creative poem about prompt engineering", is_strictly_deterministic=False, needs_live_data=False, needs_to_take_actions=False, is_open_ended_multi_step=False),
    Requirement(name="12. Answer a user's question, searching the web only if needed", is_strictly_deterministic=False, needs_live_data=True, needs_to_take_actions=True, is_open_ended_multi_step=True), # The 'search if needed' implies a reasoning loop
]

print("--- Architecture Decision Matrix Results ---\n")
for req in scenarios:
    recommendation = select_architecture(req)
    print(f"{req.name}\n  -> 🏗️  {recommendation.value}\n")


## Conclusion

By following this matrix, you avoid building expensive, slow, and unreliable Agents when a simple prompt or standard Python code would have worked perfectly.